# Wikipedia Biographical Extraction (LLM)

This notebook extracts structured biographical information from the scapred information of the Wikipedia pages and creates a final table for scored MEPs.

Run from top to bottom. Paths are configured once in the next cell for portability and easier GitHub reuse.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from difflib import SequenceMatcher
import json
import os
import re
import sys
import time
import unicodedata
import pandas as pd
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
from rapidfuzz import fuzz, process
from getpass import getpass


for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists() and (candidate / "README.md").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from src.config import PROJECT_ROOT, ensure_default_directories

ensure_default_directories()

WIKIPEDIA_DIR = PROJECT_ROOT / "3 Wikipedia"
LDA_DIR = PROJECT_ROOT / "4 LDA Topic Analysis"
LLM_WIKIPEDIA_DIR = PROJECT_ROOT / "5 LLM Wikipedia analysis"
OUTPUTS_DIR = LLM_WIKIPEDIA_DIR / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"

for folder in [OUTPUTS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

INPUT_CANDIDATES = [
    WIKIPEDIA_DIR / "outputs" / "tables" / "mep_wikipedia_final.xlsx",
    WIKIPEDIA_DIR / "mep_wikipedia_final.xlsx",
]
POLARISATION_CANDIDATES = [
    LDA_DIR / "outputs" / "tables" / "polarisation_scores_scored_only.csv",
    LDA_DIR / "polarisation_scores_scored_only.csv",
]

INPUT_PATH = next((p for p in INPUT_CANDIDATES if p.exists()), INPUT_CANDIDATES[0])
POLARISATION_SCORED_ONLY_PATH = next((p for p in POLARISATION_CANDIDATES if p.exists()), POLARISATION_CANDIDATES[0])

OUTPUT_PATH = TABLES_DIR / "wikipedia_extracted.xlsx"
PARTIAL_PATH = TABLES_DIR / "wikipedia_extracted_partial.xlsx"
FINAL_POLARISATION_PATH = TABLES_DIR / "extracted_polarisation_final.xlsx"
FINAL_POLARISATION_MANUAL_PATH = LLM_WIKIPEDIA_DIR / "extracted_polarisation_final_manually_improved_1906.xlsx"
AUDIT_OUTPUT_PATH = TABLES_DIR / "polarisation_vs_extracted_match_audit.csv"
LLM_AUDIT_JSONL_PATH = TABLES_DIR / "llm_extraction_audit.jsonl"

assert INPUT_PATH.exists(), f"Wikipedia input missing: {INPUT_PATH}"
assert POLARISATION_SCORED_ONLY_PATH.exists(), (
    f"Polarisation input missing: {POLARISATION_SCORED_ONLY_PATH}"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input file:   {INPUT_PATH}")
print(f"Output file:  {OUTPUT_PATH}")

Project root: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics
Input file:   C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\3 Wikipedia\outputs\tables\mep_wikipedia_final.xlsx
Output file:  C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\5 LLM Wikipedia analysis\outputs\tables\wikipedia_extracted.xlsx


## API key setup

This cell checks whether an OpenRouter API key is already available in the notebook environment.  
If no key is found, it asks the user to enter one securely using `getpass`.

The key is then stored only for the current notebook session as an environment variable.  
This avoids hardcoding the API key directly in the notebook.

In [ ]:
# -------------------
# API key setup
# -------------------

# Check whether an OpenRouter API key is already available.
API_KEY = (
    (os.environ.get("OPENROUTER_API_KEY") or "").strip()
    or (os.environ.get("OPENROUTER_KEY") or "").strip()
)

# If no API key is found, ask the user to enter it securely.
# getpass hides the input instead of displaying it in the notebook.
if not API_KEY:
    API_KEY = getpass("Enter your OpenRouter API key: ").strip()

    # Store the key as an environment variable for this notebook session.
    os.environ["OPENROUTER_API_KEY"] = API_KEY

# Confirm whether the key is now available without printing the key itself.
if API_KEY:
    print("OpenRouter API key is set for this session.")
else:
    print("No API key provided. Extraction cells will not run.")

No API key provided. Extraction cells will not run.


## 1 Setup: configuration, helpers, and LLM extraction functions

This cell prepares everything needed before running the actual extraction.

The cell defines the allowed classification buckets used by the model. These buckets keep the extracted variables consistent across all politicians.

Several helper functions are included:

- converting Wikipedia HTML into clean plain text
- safely parsing JSON returned by the model
- detecting whether a text refers to a Member of the European Parliament
- normalising politician names for matching across files
- flattening nested model outputs into dataframe columns
- checking whether required extraction fields are missing
- writing audit entries to a JSONL file

Finally, the cell defines the LLM extraction logic. It builds the extraction prompt, selects available free OpenRouter models, calls the model, retries failed attempts, and tries to repair invalid JSON outputs when possible.

In short, this cell sets up the full extraction engine but does not yet process the dataset.

In [12]:
# -------------------
# Allowed buckets
# -------------------

ALLOWED = {
    "gender_bucket": ["Male", "Female", "Other or Not stated"],
    "region_bucket": [
        "Northern Europe",
        "Western Europe",
        "Southern Europe",
        "Central & Eastern Europe",
        "Baltic",
    ],
    "education_level_bucket": [
        "No higher education listed",
        "Bachelor",
        "Master",
        "Doctorate",
    ],
    "education_field_bucket": [
        "Economics or Business",
        "Law",
        "Political Science or Public Administration",
        "Sociology or Demography or Social Sciences",
        "Humanities",
        "STEM",
        "Medicine or Health",
        "Education or Teaching",
        "Other or Unclear",
    ],
    "professional_background_bucket": [
        "Business or Finance",
        "Law",
        "Academia or Research",
        "Public Administration",
        "NGO or Civil Society",
        "Trade Union",
        "Journalism or Media",
        "Education Sector",
        "Health Sector",
        "Manual or Technical",
        "Career Politician",
        "Other or Unclear",
    ],
    "religion_bucket": ["Christian", "Other religion", "Explicitly secular", "Not mentioned"],
    "parental_status_bucket": ["Has children", "No children", "Not mentioned"],
    "migration_background_bucket": ["Born abroad", "Parent foreign-born", "None mentioned"],
    "international_experience_bucket": [
        "Studied abroad",
        "Worked abroad",
        "EU institutional career",
        "None significant mentioned",
    ],
    "military_background_bucket": ["Yes", "No", "Not mentioned"],
    "prior_political_level_bucket": [
        "Local only",
        "National parliament",
        "Government minister",
        "No prior elected office",
        "Unclear",
    ],
    "demography_expertise_bucket": ["Demography specialist", "Social policy background", "None"],
}


REQUIRED_EXTRACT_FIELDS = [
    "full_name",
    "birth_date",
    "country_of_origin",
    "buckets",
]


SYSTEM = (
    "You extract and classify structured biographical info from Wikipedia text. "
    "Return only valid JSON that matches the requested schema exactly. "
    "Use ONLY the allowed bucket values. "
    "If unknown / not stated in the text, use null, or empty list where applicable. "
    "Do not invent details. "
    "For classifications, choose the closest bucket only when the text supports it. "
    "Always provide short evidence snippets copied from the text, max around 200 chars each."
)


# -------------------
# General helpers
# -------------------

def html_to_text(html: str) -> str:
    """Convert raw HTML into clean plain text."""

    if not isinstance(html, str) or not html.strip():
        return ""

    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def safe_json_loads(s: str) -> dict:
    """Parse JSON safely, including outputs wrapped in markdown fences."""

    s = s.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)

    match = re.search(r"\{.*\}", s, flags=re.DOTALL)
    if match:
        s = match.group(0).strip()

    try:
        return json.loads(s)
    except json.JSONDecodeError:
        s2 = re.sub(r",\s*([}\]])", r"\1", s)
        s2 = s2.replace("\u00a0", " ").strip()
        return json.loads(s2)


def is_mep_from_text(text: str) -> bool:
    """Check whether the text suggests European Parliament membership."""

    if not text:
        return False

    patterns = [
        r"\bMember of the European Parliament\b",
        r"\bMEP\b",
        r"\bMembers of the European Parliament\b",
        r"\bEuropean Parliament\b.*\bmember\b",
        r"\b(member|deputy)\b.*\bEuropean Parliament\b",
    ]

    text_flat = " ".join(text.split())

    return any(re.search(p, text_flat, flags=re.IGNORECASE) for p in patterns)


def norm_name(x: str) -> str:
    """Normalise names for matching across files."""

    if not isinstance(x, str):
        return ""

    x = "".join(
        c for c in unicodedata.normalize("NFKD", x)
        if not unicodedata.combining(c)
    )

    x = x.lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    return x


def match_key(x: str) -> str:
    """Create a stable name matching key."""

    stop_tokens = {"mr", "mrs", "ms", "dr", "prof", "sir", "jr", "sr", "mep"}

    tokens = [
        t for t in norm_name(x).split()
        if t and t not in stop_tokens
    ]

    return " ".join(sorted(tokens))


def flatten_extract(d: dict) -> dict:
    """Flatten nested model output into dataframe columns."""

    out = {}

    out["full_name_extracted"] = d.get("full_name")
    out["birth_date_extracted"] = d.get("birth_date")
    out["place_of_birth_extracted"] = d.get("place_of_birth")
    out["country_of_origin_extracted"] = d.get("country_of_origin")
    out["ep_group_extracted"] = d.get("ep_group")
    out["mep_since_extracted"] = d.get("mep_since")

    out["occupations_extracted"] = json.dumps(d.get("occupations", []), ensure_ascii=False)
    out["parties_extracted"] = json.dumps(d.get("parties", []), ensure_ascii=False)
    out["education_extracted"] = json.dumps(d.get("education", []), ensure_ascii=False)

    out["confidence_extracted"] = d.get("confidence")

    buckets = d.get("buckets", {}) or {}
    for key, value in buckets.items():
        out[key] = value

    evidence = d.get("evidence", {}) or {}
    for key, value in evidence.items():
        out[f"evidence_{key}"] = value

    return out


def extraction_quality_flags(parsed: dict) -> dict:
    """Check whether the model output contains required fields."""

    if not isinstance(parsed, dict):
        return {
            "malformed_output": True,
            "missing_required_fields": REQUIRED_EXTRACT_FIELDS,
        }

    missing = [
        k for k in REQUIRED_EXTRACT_FIELDS
        if k not in parsed or parsed.get(k) in [None, "", []]
    ]

    return {
        "malformed_output": False,
        "missing_required_fields": missing,
    }


def append_audit_jsonl(entry: dict) -> None:
    """Append one audit record to the JSONL audit log."""

    LLM_AUDIT_JSONL_PATH.parent.mkdir(parents=True, exist_ok=True)

    with open(LLM_AUDIT_JSONL_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")


# -------------------
# Model selection and LLM call
# -------------------

STATIC_MODEL_CANDIDATES = [
    "meta-llama/llama-3.3-70b-instruct:free",
    "openai/gpt-oss-20b:free",
    "qwen/qwen3-coder:free",
    "google/gemma-4-31b-it:free",
    "meta-llama/llama-3.2-3b-instruct:free",
]


def discover_live_free_models(limit: int = LIVE_FREE_LIMIT) -> list[str]:
    """Fetch currently available free OpenRouter model IDs."""

    if not API_KEY:
        return []

    try:
        resp = requests.get(
            f"{BASE_URL}/models",
            headers={"Authorization": f"Bearer {API_KEY}"},
            timeout=30,
        )
        resp.raise_for_status()

        data = resp.json().get("data", [])
        ids = [m.get("id", "") for m in data if isinstance(m, dict)]
        ids = [m for m in ids if m.endswith(":free")]

        return ids[:limit]

    except Exception:
        return []


def build_model_candidates() -> list[str]:
    """Combine static and live free models without duplicates."""

    live = discover_live_free_models()

    merged = []
    for m in STATIC_MODEL_CANDIDATES + live:
        m = (m or "").strip()
        if m and m not in merged:
            merged.append(m)

    return merged


MODEL_CANDIDATES = build_model_candidates()


def is_non_retriable_model_error(err_text: str) -> bool:
    """Identify errors where retrying the same model is pointless."""

    t = err_text.lower()

    return (
        "error code: 404" in t
        or "no endpoints found" in t
        or "unavailable for free" in t
        or "model not found" in t
    )


def build_extraction_prompt(raw_text: str, name_hint: str, lang: str) -> str:
    """Build the extraction prompt sent to the model."""

    return f"""
Name hint: {name_hint}
Language of page: {lang}

Task:
Return ONLY valid JSON matching the schema. No commentary.

Allowed buckets, exact strings:
{json.dumps(ALLOWED, ensure_ascii=False, indent=2)}

JSON schema:
{{
  "full_name": string|null,
  "birth_date": string|null,
  "place_of_birth": string|null,
  "country_of_origin": string|null,
  "education": [{{"degree": string|null,"field": string|null,"institution": string|null,"country": string|null,"start_year": integer|null,"end_year": integer|null,"notes": string|null}}],
  "occupations": [string],
  "parties": [string],
  "ep_group": string|null,
  "mep_since": string|null,
  "buckets": {{
    "gender_bucket": string|null,
    "region_bucket": string|null,
    "education_level_bucket": string|null,
    "education_field_bucket": string|null,
    "professional_background_bucket": string|null,
    "religion_bucket": string|null,
    "parental_status_bucket": string|null,
    "migration_background_bucket": string|null,
    "international_experience_bucket": string|null,
    "military_background_bucket": string|null,
    "prior_political_level_bucket": string|null,
    "demography_expertise_bucket": string|null
  }},
  "confidence": number,
  "evidence": {{
    "birth_date": string|null,
    "country_of_origin": string|null,
    "ep_group": string|null,
    "education_level_bucket": string|null,
    "education_field_bucket": string|null,
    "professional_background_bucket": string|null,
    "parental_status_bucket": string|null,
    "migration_background_bucket": string|null,
    "international_experience_bucket": string|null,
    "prior_political_level_bucket": string|null
  }}
}}

Wikipedia text:
\"\"\"{raw_text[:MAX_INPUT_CHARS]}\"\"\"
""".strip()


def call_model(model_name: str, prompt: str):
    """Call one model once."""

    if client is None:
        raise RuntimeError("OPENROUTER_API_KEY is not set.")

    return client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=MAX_OUTPUT_TOKENS,
    )


def extract_with_llm(raw_text: str, name_hint: str, lang: str) -> dict:
    """Extract structured biographical information using model fallback."""

    global LAST_MODEL_USED

    if client is None:
        raise RuntimeError("OPENROUTER_API_KEY is not set.")

    if not MODEL_CANDIDATES:
        raise RuntimeError("No free model candidates available right now.")

    prompt = build_extraction_prompt(raw_text, name_hint, lang)
    errors = []

    for model_name in MODEL_CANDIDATES:
        for attempt in range(1, MAX_RETRIES_PER_MODEL + 1):
            try:
                resp = call_model(model_name, prompt)
                content = (resp.choices[0].message.content or "").strip()

                if not content:
                    raise ValueError("Empty model output")

                try:
                    parsed = safe_json_loads(content)
                    LAST_MODEL_USED = model_name
                    return parsed

                except Exception:
                    fixer = f"""
Fix the following so it is strictly valid JSON.
Do not change meaning or values.
Return JSON only.

INPUT:
{content}
""".strip()

                    resp2 = client.chat.completions.create(
                        model=model_name,
                        messages=[
                            {"role": "system", "content": "Return only valid JSON."},
                            {"role": "user", "content": fixer},
                        ],
                        temperature=0.0,
                        max_tokens=MAX_OUTPUT_TOKENS,
                    )

                    fixed = (resp2.choices[0].message.content or "").strip()
                    parsed = safe_json_loads(fixed)

                    LAST_MODEL_USED = model_name
                    return parsed

            except Exception as e:
                msg = str(e)

                errors.append(f"{model_name} attempt {attempt}: {msg}")

                if is_non_retriable_model_error(msg):
                    break

                time.sleep(BACKOFF_SECONDS * attempt)

    short_errors = "\n".join(errors[-8:])
    raise RuntimeError("All free model attempts failed. Last errors:\n" + short_errors)


print(f"Prompt version: {PROMPT_VERSION}")
print(f"Model candidates: {len(MODEL_CANDIDATES)}")
print(f"Audit log path: {LLM_AUDIT_JSONL_PATH}")

Prompt version: v1.0.0
Model candidates: 5
Audit log path: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\5 LLM Wikipedia analysis\outputs\tables\llm_extraction_audit.jsonl


## 2 Run extraction: reuse existing results and process missing names

This cell checks whether new LLM extraction is actually needed, then runs extraction only for missing politicians.

It first loads the scored polarisation dataset and creates standardised name keys for each MEP. These keys are used to match names reliably across files, even if spelling, titles, punctuation, or accents differ slightly.

The cell then checks existing output files, including partial and final extraction files. If previous extracted results already exist, they are loaded and compared against the target MEP names.

If all required MEPs are already covered, the cell stops and does not call the LLM again. This avoids wasting API calls and time.

If some MEPs are missing, the cell loads the Wikipedia source file, selects only the missing names, and runs the LLM extraction for those rows. For each person, it converts the Wikipedia HTML into plain text, sends it to the model, flattens the result into table columns, checks extraction quality, and marks rows that need manual review.

Every extraction attempt is also written to an audit log with the timestamp, model used, prompt version, error status, raw response, parsed output, missing fields, and review flag.

At the end, the cell saves two outputs: a partial file containing only the newly extracted rows, and a combined output file containing existing results plus the new extractions.

In [16]:
# ============================================================
# RUN: check existing extraction files and extract only missing
# ============================================================

# -------------------
# Load scored MEP target names
# -------------------

pol_df = pd.read_csv(POLARISATION_SCORED_ONLY_PATH).copy()

pol_name_candidates = ["mep_name_raw", "mep_name_clean", "name"]
pol_name_col = next((c for c in pol_name_candidates if c in pol_df.columns), None)

if pol_name_col is None:
    raise ValueError(
        f"Could not find a name column in polarisation file. "
        f"Columns: {list(pol_df.columns)}"
    )

pol_df["_name_key"] = pol_df[pol_name_col].fillna("").astype(str).map(match_key)
pol_df = pol_df[pol_df["_name_key"] != ""].copy()
pol_df = pol_df.drop_duplicates(subset=["_name_key"], keep="first")

target_keys = set(pol_df["_name_key"].tolist())


# -------------------
# Load existing extraction files
# -------------------

reuse_candidates = [
    PARTIAL_PATH,
    FINAL_POLARISATION_PATH,
    FINAL_POLARISATION_MANUAL_PATH,
    OUTPUT_PATH,
]

reuse_frames = []
reuse_paths_used = []

for candidate_path in reuse_candidates:
    if candidate_path.exists():
        try:
            tmp_df = pd.read_excel(candidate_path)

            if not tmp_df.empty:
                reuse_frames.append(tmp_df)
                reuse_paths_used.append(str(candidate_path))

        except Exception as e:
            print(f"Could not read reuse candidate {candidate_path}: {e}")

if reuse_frames:
    reuse_df = pd.concat(reuse_frames, ignore_index=True)
else:
    reuse_df = pd.DataFrame()

# -------------------
# Identify already processed names using all possible name columns
# -------------------

reuse_name_candidates = [
    "mep_name_raw",
    "mep_name_clean",
    "name",
    "full_name_extracted",
]

processed_keys = set()

if not reuse_df.empty:
    available_name_cols = [
        c for c in reuse_name_candidates
        if c in reuse_df.columns
    ]

    if not available_name_cols:
        print("No usable name columns found in reuse files.")
    else:
        print(f"Using reuse name columns: {available_name_cols}")

        for col in available_name_cols:
            keys = (
                reuse_df[col]
                .fillna("")
                .astype(str)
                .map(match_key)
            )

            processed_keys.update(keys[keys != ""].tolist())

else:
    processed_keys = set()


# IMPORTANT: recalculate missing keys after processed_keys has been updated
missing_keys = sorted([k for k in target_keys if k not in processed_keys])
LLM_NEEDED_NOW = len(missing_keys) > 0

# -------------------
# Stop here if no extraction is needed
# -------------------

if not LLM_NEEDED_NOW:
    print("No new LLM extraction needed. Existing extracted files already cover the scored MEPs.")

else:
    if client is None:
        raise RuntimeError(
            "Missing names require new extraction, but OPENROUTER_API_KEY is not set."
        )

    # -------------------
    # Load Wikipedia source file
    # -------------------

    source_df = pd.read_excel(INPUT_PATH).copy()

    source_name_candidates = ["name", "mep_name_raw", "mep_name_clean", "full_name_extracted"]
    source_name_col = next((c for c in source_name_candidates if c in source_df.columns), None)

    if source_name_col is None:
        raise ValueError(
            f"Could not find a name column in source file. "
            f"Columns: {list(source_df.columns)}"
        )

    source_df["_name_key"] = source_df[source_name_col].fillna("").astype(str).map(match_key)

    df_run = source_df[source_df["_name_key"].isin(missing_keys)].copy()

    if N_FIRST is not None:
        df_run = df_run.head(N_FIRST).copy()

    print(f"Rows selected for new extraction: {len(df_run)}")

    if df_run.empty:
        raise RuntimeError(
            "Missing names were found, but none matched rows in the Wikipedia source file."
        )

    # -------------------
    # Run extraction
    # -------------------

    rows = []
    audit_rows = []

    for idx, row in df_run.iterrows():
        name_hint = str(row.get(source_name_col, "") or "")
        lang = str(row.get("wikipedia_language", "") or "")

        html_en = row.get("wikipedia_content_html_en")
        html_local = row.get("wikipedia_content_html")

        text_en = html_to_text(html_en) if isinstance(html_en, str) else ""
        text_local = html_to_text(html_local) if isinstance(html_local, str) else ""

        text = text_en + "\n\n" + text_local

        ts = datetime.now(timezone.utc).isoformat()
        status = "ok"
        err_msg = None
        extracted = None

        try:
            extracted = extract_with_llm(text, name_hint=name_hint, lang=lang)
            flat = flatten_extract(extracted)

            mep_flag = (
                bool(extracted.get("mep_since") or extracted.get("ep_group"))
                or is_mep_from_text(text)
            )

            flat["MEP"] = bool(mep_flag)

            quality = extraction_quality_flags(extracted)

            flat["needs_manual_review"] = bool(
                quality["malformed_output"]
                or len(quality["missing_required_fields"]) > 0
            )

            flat["missing_required_fields"] = json.dumps(
                quality["missing_required_fields"],
                ensure_ascii=False,
            )

            flat["extract_error"] = None

        except Exception as e:
            status = "error"
            err_msg = str(e)

            flat = {
                "MEP": pd.NA,
                "extract_error": err_msg,
                "needs_manual_review": True,
                "missing_required_fields": json.dumps(
                    REQUIRED_EXTRACT_FIELDS,
                    ensure_ascii=False,
                ),
            }

        rows.append(flat)

        audit_entry = {
            "timestamp_utc": ts,
            "name": name_hint,
            "wikipedia_language": lang,
            "model_used": globals().get("LAST_MODEL_USED", MODEL),
            "prompt_version": PROMPT_VERSION,
            "error_status": status,
            "error_message": err_msg,
            "raw_response": None if extracted is None else json.dumps(extracted, ensure_ascii=False),
            "parsed_output": None if extracted is None else extracted,
            "missing_required_fields": (
                REQUIRED_EXTRACT_FIELDS
                if extracted is None
                else extraction_quality_flags(extracted)["missing_required_fields"]
            ),
            "needs_manual_review": bool(
                status != "ok"
                or (
                    extracted is not None
                    and len(extraction_quality_flags(extracted)["missing_required_fields"]) > 0
                )
            ),
        }

        append_audit_jsonl(audit_entry)
        audit_rows.append(audit_entry)

    # -------------------
    # Save new and combined outputs
    # -------------------

    df_new = pd.concat(
        [
            df_run.reset_index(drop=True),
            pd.DataFrame(rows),
        ],
        axis=1,
    )

    if reuse_df.empty:
        df_combined = df_new.copy()
    else:
        df_combined = pd.concat([reuse_df, df_new], ignore_index=True)

    if "_name_key" not in df_combined.columns:
        name_col = next(
            (c for c in ["name", "full_name_extracted", "mep_name_raw", "mep_name_clean"] if c in df_combined.columns),
            None,
        )

        if name_col is not None:
            df_combined["_name_key"] = df_combined[name_col].fillna("").astype(str).map(match_key)

    if "_name_key" in df_combined.columns:
        df_combined = df_combined.drop_duplicates(subset=["_name_key"], keep="last")

    df_new.to_excel(PARTIAL_PATH, index=False)
    df_combined.to_excel(OUTPUT_PATH, index=False)

    assert len(df_new) > 0, "New extraction output is empty"
    assert PARTIAL_PATH.exists(), f"Missing partial output file: {PARTIAL_PATH}"
    assert OUTPUT_PATH.exists(), f"Missing output file: {OUTPUT_PATH}"

    print(f"Wrote new extraction rows: {PARTIAL_PATH}")
    print(f"Wrote combined extraction output: {OUTPUT_PATH}")
    print(f"Audit rows written: {len(audit_rows)} -> {LLM_AUDIT_JSONL_PATH}")

Using reuse name columns: ['mep_name_raw', 'mep_name_clean', 'name', 'full_name_extracted']
No new LLM extraction needed. Existing extracted files already cover the scored MEPs.


## Final manual adjustments

After the automated extraction workflow, I manually reviewed and corrected a small number of country irregularities.
